# Example - Effectiveness of Vaccine (Li and Pearl, 2024)

In this notebook, the results of the "Effectiveness of Vaccine" example are replicated.

References:

> **"Probabilities of Causation with Nonbinary Treatment and Effect"**
> 
> https://ojs.aaai.org/index.php/AAAI/article/view/30030


> **"Probabilities of Causation with Nonbinary Treatment and Effect"**
> 
> arXiv:2208.09568v1
> 
> https://arxiv.org/pdf/2208.09568

In [1]:
import sys
sys.path.insert(0, '../src/')

import numpy as np

from probabilities_of_causation import *

## Treatments and outcomes 

In [2]:
treatment = {'x1': 'vaccinated', 'x2': 'unvaccinated'}
outcome = {'y1': 'uninfected', 'y2': 'asymptomatic', 'y3': 'mild symptoms', 'y4': 'severe condition'} 

## Experimental and observational data

See paper `Table 5: Experimental data of the clinical study`.

|               | x1: Vaccinated | x2: Unvaccinated |
|---------------|------------|--------------|
| y1: Uninfected    | 205 | 27 |
| y2: Asymptomatic  | 46 | 122 |
| y3: Mild Symptoms | 343 | 87 |
| y4: Severe Condition | 6 | 364 |
| Overall       | 600 | 600 |

In [3]:
experimental_data = {
    ('y1', 'x1'): 205/600,
    ('y2', 'x1'): 46/600,
    ('y3', 'x1'): 343/600,
    ('y4', 'x1'): 6/600,
    ('y1', 'x2'): 27/600,
    ('y2', 'x2'): 122/600,
    ('y3', 'x2'): 87/600,
    ('y4', 'x2'): 364/600
}

See paper `Table 6: Observational data of the clinical study`. 

|               | x1: Vaccinated | x2: Unvaccinated |
|---------------|------------|--------------|
| y1: Uninfected    | 6 | 52 |
| y2: Asymptomatic  | 74 | 243 |
| y3: Mild Symptoms | 632 | 147 |
| y4: Severe Condition | 5 | 41 |
| Overall       | 717 | 483 |

In [4]:
observational_data = {
    ('x1', 'y1'): 6/1200,
    ('x1', 'y2'): 74/1200,
    ('x1', 'y3'): 632/1200,
    ('x1', 'y4'): 5/1200,
    ('x2', 'y1'): 52/1200,
    ('x2', 'y2'): 243/1200,
    ('x2', 'y3'): 147/1200,
    ('x2', 'y4'): 41/1200
}

In [5]:
probabilities = Probabilities(observational_data, experimental_data)


=== Data Completeness Check ===

✓ All observational probabilities P(x, y) are present

✓ All experimental probabilities P(y|do(x)) are present

=== Marginal Probability Check ===
P(x1) = 0.5975
P(x2) = 0.4025
P(y1) = 0.0483
P(y2) = 0.2642
P(y3) = 0.6492
P(y4) = 0.0383
=== End of Data Completeness Check ===



## Queries and verification of bounds

The clinical study aims to identify the following probabilities: 

- probability that a patient would be infected in a severe condition (y4) if unvaccinated (x1) and would be uninfected (y1) if vaccinated (x1)
- probability that a patient would be infected in a severe condition (y4) if unvaccinated (x2) and would be asymptomatic infected (y2) if vaccinated (x1)
- probability that a patient would be infected in a severe condition (y4) if unvaccinated and would be infected with mild symptoms (y3) if vaccinated (x1)


The goals of the clinical study are expressed as the following probabilities of causation:
- $P({y_1}_{x_1},{y_4}_{x_2})$
- $P({y_2}_{x_1},{y_4}_{x_2})$
- $P({y_3}_{x_1},{y_4}_{x_2})$

### Query $P({y_1}_{x_1},{y_4}_{x_2})$

In [6]:
query = {
    "hypotheticals": [('y1', 'x1'), ('y4', 'x2')],
    "x_p": None,
    "y_q": None
}

In [7]:
lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.000
Upper bound: 0.039 


In [8]:
hypotheticals = query.get("hypotheticals", [])
lb, ub = prob_necessity_sufficiency_k(hypotheticals, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Lower bound: 0.000
Upper bound: 0.039 


According to the paper:

$0 \le P({y_1}_{x_1},{y_4}_{x_2}) \le 0.039$

In [9]:
np.isclose(0, lb, atol=1e-3)

np.True_

In [10]:
np.isclose(0.039, ub, atol=1e-3)

np.True_

### Query $P({y_2}_{x_1},{y_4}_{x_2})$

In [11]:
query = {
    "hypotheticals": [('y2', 'x1'), ('y4', 'x2')],
    "x_p": None,
    "y_q": None
}

In [12]:
lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.037
Upper bound: 0.077 


According to the paper:

$0.037 \le P({y_2}_{x_1},{y_4}_{x_2}) \le 0.077$

In [13]:
np.isclose(0.037, lb, atol=1e-3)

np.True_

In [14]:
np.isclose(0.077, ub, atol=1e-3)

np.True_

### Query $P({y_3}_{x_1},{y_4}_{x_2})$

In [15]:
query = {
    "hypotheticals": [('y3', 'x1'), ('y4', 'x2')],
    "x_p": None,
    "y_q": None
}

In [16]:
lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 8: probability of necessity and sufficiency
Lower bound: 0.502
Upper bound: 0.561 


According to the paper:

$0.502 \le P({y_3}_{x_1},{y_4}_{x_2}) \le 0.561$

In [17]:
np.isclose(0.502, lb, atol=1e-3)

np.True_

In [18]:
np.isclose(0.561, ub, atol=1e-3)

np.True_

##  Intermediate calculation steps reported in the paper
See `Appendix/Calculation in the Examples/Efectiveness of Vaccine`

> https://arxiv.org/pdf/2208.09568

### By Theorem 7:

$\large  0\le P({y_4}_{x_2},x_1,y_1)\le 0.005 $

In [19]:
query = {
    "hypotheticals": [('y4', 'x2')],
    "x_p": 'x1',
    "y_q": 'y1'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.000
Upper bound: 0.005 


### By Theorem 7:

$\large 0\le P({y_1}_{x_1},x_2,y_4)\le 0.034 $

In [20]:
query = {
    "hypotheticals": [('y1', 'x1')],
    "x_p": 'x2',
    "y_q": 'y4'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.000
Upper bound: 0.034 


### By Theorem 7:

$\large 0.037\le P({y_4}_{x_2},x_1,y_2)\le 0.062 $

In [21]:
query = {
    "hypotheticals": [('y4', 'x2')],
    "x_p": 'x1',
    "y_q": 'y2'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.037
Upper bound: 0.062 


### By Theorem 7:

$\large 0\le P({y_2}_{x_1},x_2,y_4)\le 0.015$

In [22]:
query = {
    "hypotheticals": [('y2', 'x1')],
    "x_p": 'x2',
    "y_q": 'y4'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.000
Upper bound: 0.015 


### By Theorem 7:

$\large 0.502\le P({y_4}_{x_2},x_1,y_3)\le 0.527 $

In [23]:
query = {
    "hypotheticals": [('y4', 'x2')],
    "x_p": 'x1',
    "y_q": 'y3'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.502
Upper bound: 0.527 


### By Theorem 7:

$\large  0\le P({y_3}_{x_1},x_2,y_4)\le 0.034 $

In [24]:
query = {
    "hypotheticals": [('y3', 'x1')],
    "x_p": 'x2',
    "y_q": 'y4'
}

lb, ub = get_bounds(query, probabilities)
print(f"Lower bound: {lb:.3f}\nUpper bound: {ub:.3f} ")

Applying theorem 7: probability of necessity
Lower bound: 0.000
Upper bound: 0.034 
